In [1]:
import numpy as np
import pandas as pd
import plotly.express as px

from rfm_sp500 import constants

In [2]:
weekly_returns = pd.read_csv(
    constants.PROCESSED_DATA_DIR / "weekly_returns.csv.gz", index_col="Date", parse_dates=True
)
benchmark_returns = pd.read_csv(
    constants.PROCESSED_DATA_DIR / "benchmark_weekly_returns.csv.gz",
    index_col="Date",
    parse_dates=True,
)

daily = pd.read_csv(constants.PROCESSED_DATA_DIR / "sp500_clean_daily.csv.gz", parse_dates=["Date"])
daily = daily.dropna(subset=["Close"]).sort_values(["Ticker", "Date"])

print(weekly_returns.shape, benchmark_returns.shape)

(207, 494) (207, 1)


Mean weekly return

In [3]:
mean_return = weekly_returns.mean().rename("Mean_Weekly_Return")
mean_return.describe()

count    494.000000
mean       0.002928
std        0.002726
min       -0.004468
25%        0.001216
50%        0.002675
75%        0.004304
max        0.017772
Name: Mean_Weekly_Return, dtype: float64

**From Share clustering Tutorial

In [4]:
def calculate_weekly_beta(share_returns, market_returns):
    paired = pd.concat(
        [share_returns.rename("Share"), market_returns.rename("Market")], axis=1
    ).dropna()
    beta = paired["Share"].cov(paired["Market"]) / paired["Market"].var()
    return beta, len(paired)


results = []
for ticker in weekly_returns.columns:
    beta, n = calculate_weekly_beta(weekly_returns[ticker], benchmark_returns["Adj Close"])
    results.append({"Share": ticker, "Beta": beta, "Paired_weeks": n})

beta_df = pd.DataFrame(results).set_index("Share")
beta_df.describe()

,Beta,Paired_weeks
count,494.000000,494.000000
mean,0.979350,206.008097
std,0.420169,9.975626
min,0.055453,64.000000
25%,0.667784,207.000000
50%,0.950610,207.000000
75%,1.179538,207.000000
max,3.778560,207.000000


Independently checked evidence (** slide 11 and 13)

In [5]:
# Lecture worked example (slide 11): should give beta 2.0 from 3 weeks
print(calculate_weekly_beta(pd.Series([-0.04, 0, 0.04]), pd.Series([-0.02, 0, 0.02])))

# Beta = slope of share returns against market returns (slide 13)
paired = pd.concat([weekly_returns["AAPL"], benchmark_returns], axis=1).dropna()
print(beta_df.loc["AAPL", "Beta"], np.polyfit(paired.iloc[:, 1], paired.iloc[:, 0], 1)[0])

(np.float64(2.0), 3)
1.1748359849046572 1.1748359849046575


In [6]:
tickers = pd.read_csv(constants.RAW_DATA_DIR / "sp500_tickers.csv.gz").set_index("Symbol")
beta_df.join(tickers[["Security", "GICS Sector"]]).sort_values("Beta").iloc[[0, 1, 2, -3, -2, -1]]

,Beta,Paired_weeks,Security,GICS Sector
Share,,,,
GIS,0.055453,207,General Mills,Consumer Staples
SJM,0.175579,207,J.M. Smucker Company (The),Consumer Staples
MRK,0.198288,207,Merck & Co.,Health Care
XYZ,2.504520,207,"Block, Inc.",Financials
COIN,3.004326,193,Coinbase,Financials
CVNA,3.778560,207,Carvana,Consumer Discretionary


In [7]:
last_dates = daily.groupby("Ticker")["Date"].max()
obs_date = last_dates.min()
print(
    "Observation date:",
    obs_date.date(),
    ", all shares end on date?:",
    (last_dates == obs_date).all(),
)

Observation date: 2024-12-31 , all shares end on date?: True


Taken from RFM_Yahoofin seminar

In [8]:
def calculate_rfm(data, obs_date):
    data = data[data["Date"] <= obs_date].set_index("Date")

    # Average of last 20 days, excluding the current day
    average_volume = data["Volume"].shift(1).rolling(20).mean()
    data["HighVolume"] = data["Volume"] > 2 * average_volume

    # Days with a complete 20-day baseline
    valid = data.loc[average_volume.notna()]
    if len(valid) < 60:
        return None

    recent = valid.tail(60)
    events = valid.index[valid["HighVolume"]]

    return {
        "Recency": (obs_date - events.max()).days if len(events) else np.nan,
        "Frequency": int(recent["HighVolume"].sum()),
        "Monetary": (recent["Close"] * recent["Volume"]).sum(),
    }


rfm_results = {}
for ticker, data in daily.groupby("Ticker"):
    rfm = calculate_rfm(data, obs_date)
    if rfm is not None:
        rfm_results[ticker] = rfm

rfm_df = pd.DataFrame.from_dict(rfm_results, orient="index")
print(len(rfm_df))
print("Missing Recency:", rfm_df["Recency"].isna().sum())
rfm_df.describe()

494
Missing Recency: 0


,Recency,Frequency,Monetary
count,494.000000,494.000000,4.940000e+02
mean,20.615385,2.993927,3.970149e+10
std,21.400599,1.327645,1.283828e+11
min,4.000000,0.000000,1.297483e+09
25%,11.000000,2.000000,1.025102e+10
50%,11.000000,3.000000,1.688961e+10
75%,18.000000,4.000000,3.262167e+10
max,222.000000,8.000000,1.847697e+12


Alignment check (1.5)

In [9]:
aapl = daily[daily["Ticker"] == "AAPL"].set_index("Date")
day = aapl.index[-1]
previous_20 = aapl["Volume"].iloc[-21:-1]

print("Day:", day.date())
print("Last 20 days:", previous_20.index[0].date(), "-", previous_20.index[-1].date())
print("Threshold using mean():", 2 * previous_20.mean())
print("Threshold using code:", 2 * aapl["Volume"].shift(1).rolling(20).mean().iloc[-1])

Day: 2024-12-31
Last 20 days: 2024-12-02 - 2024-12-30
Threshold using mean(): 93843540.0
Threshold using code: 93843540.0


Worked example (1.5)

In [10]:
aapl["Threshold"] = 2 * aapl["Volume"].shift(1).rolling(20).mean()
aapl["HighVolume"] = aapl["Volume"] > aapl["Threshold"]
aapl["TradedValue"] = aapl["Close"] * aapl["Volume"]

last_60 = aapl.tail(60)
print("Window:", last_60.index[0].date(), "to", last_60.index[-1].date())
print("\nHigh-volume days in the window:")
print(last_60.loc[last_60["HighVolume"], ["Volume", "Threshold"]])

latest_event = aapl.index[aapl["HighVolume"]].max()
print("\nLast:", latest_event.date(), ", Recency:", (obs_date - latest_event).days, "days")
print("Frequency:", last_60["HighVolume"].sum())
print("Monetary: $", f"{last_60['TradedValue'].sum():,.0f}")
print("\nFrom calculate_rfm:", rfm_df.loc["AAPL"].to_dict())

Window: 2024-10-07 to 2024-12-31

High-volume days in the window:
                 Volume   Threshold
Date                               
2024-11-25   90152800.0  87641480.0
2024-12-20  147495300.0  90009210.0

Last: 2024-12-20 , Recency: 11 days
Frequency: 2
Monetary: $ 623,303,377,707

From calculate_rfm: {'Recency': 11.0, 'Frequency': 2.0, 'Monetary': 623303377706.5842}


In [11]:
tickers = pd.read_csv(constants.RAW_DATA_DIR / "sp500_tickers.csv.gz").set_index("Symbol")

features = beta_df.join(mean_return).join(rfm_df).join(tickers[["Security", "GICS Sector"]])


full = features.dropna(subset=["Mean_Weekly_Return", "Beta", "Recency", "Frequency", "Monetary"])
print("Missing sector:", full["GICS Sector"].isna().sum())
print(len(features), "shares,", len(full), "with all features")

full.to_csv(constants.PROCESSED_DATA_DIR / "features.csv.gz")
full.describe().round(4)

Missing sector: 0
494 shares, 494 with all features


,Beta,Paired_weeks,Mean_Weekly_Return,Recency,Frequency,Monetary
count,494.0000,494.0000,494.0000,494.0000,494.0000,4.940000e+02
mean,0.9794,206.0081,0.0029,20.6154,2.9939,3.970149e+10
std,0.4202,9.9756,0.0027,21.4006,1.3276,1.283828e+11
min,0.0555,64.0000,-0.0045,4.0000,0.0000,1.297483e+09
25%,0.6678,207.0000,0.0012,11.0000,2.0000,1.025102e+10
50%,0.9506,207.0000,0.0027,11.0000,3.0000,1.688961e+10
75%,1.1795,207.0000,0.0043,18.0000,4.0000,3.262167e+10
max,3.7786,207.0000,0.0178,222.0000,8.0000,1.847697e+12


In [ ]:
for col in ["Mean_Weekly_Return", "Beta", "Recency", "Frequency", "Monetary"]:
    fig = px.histogram(full, x=col, marginal="box", title=f"Distribution of {col}")
    fig.write_image(constants.FIGURES_DIR / f"{col}_distribution.png")
    fig.show()

full[["Mean_Weekly_Return", "Beta", "Recency", "Frequency", "Monetary"]].skew().round(2)

Mean_Weekly_Return     0.98
Beta                   1.26
Recency                3.56
Frequency              0.42
Monetary              11.70
dtype: float64